# 🥒 Zucchini Ultimate Pipeline — Colorspace × Ordinal-Loss Ablation → Final Model

Builds on the findings from the prior incremental-ablation notebook and adds
three new axes on top of a deeper backbone:

| Axis | What's tested | Why |
|---|---|---|
| **CNN depth** | 6-conv-block backbone (vs the earlier 4-block one) | more capacity for the harder colorspace/loss grid below |
| **Image encoding** | RGB, LAB, HSV, YCrCb | does the colorspace the CNN sees change anything? |
| **Ordinal loss** | CORAL-style (independent per-threshold BCE, used previously) vs **CORN** (conditional/filtered per-threshold BCE) | CORN only trains each threshold on samples for which it's actually defined, which should give better rank-consistency |
| **Color-feature branch** | on / off | prior study found this **consistently hurts** — re-validated here on the new backbone, then left off for the main grid |

Carried-forward findings from the prior notebook (re-validated in §13, not
re-litigated in full): **dual head + composite-style loss beats a single
head**, and **the color-feature branch hurts every time it's included**.

The final section (§19) assembles the single best configuration found here
into one clean, deployable pipeline with `reg` / `ord` / `blend` scoring and
a ready-to-use `predict_quality()` function.

**Compute note:** the full colorspace×ordinal-loss grid is 8 configs × 5
folds = 40 fold-trainings on a deeper CNN than before — expect this to take
noticeably longer than the earlier notebook. A `QUICK_MODE` toggle is
provided in §1 to shrink epochs/patience for a fast smoke-test pass; turn it
off for the real numbers.

## 0 · Imports

In [ ]:
import os, glob, random, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import tifffile as tiff
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr, kendalltau, skew, kurtosis
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                             cohen_kappa_score)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Imports OK ✔')

## 1 · Configuration

In [ ]:
# ── EDIT THIS PATH ────────────────────────────────────────────────────────────
NOCC_ROOT = r'F:\Aymen\server\statistical_filtering'
# ─────────────────────────────────────────────────────────────────────────────

QUICK_MODE = False   # True = fast smoke-test (fewer epochs/patience); False = real run

IMG_SIZE    = (128, 128)
BATCH_SIZE  = 16
EPOCHS      = 40 if QUICK_MODE else 300
K_FOLDS     = 5
PATIENCE_ES = 8  if QUICK_MODE else 30
PATIENCE_LR = 4  if QUICK_MODE else 8
LR_INIT     = 5e-4
LR_MIN      = 1e-6
LR_FACTOR   = 0.5

SCORE_FOLDERS = list(range(1, 10))
NUM_CLASSES   = 9
IDX_TO_SCORE  = np.arange(1, 10, dtype=np.float32)
FEAT_DIM      = 54
NUM_ORD       = 8            # 8 thresholds for scores 1..9

ENCODINGS      = ['RGB', 'LAB', 'HSV', 'YCrCb']       # image-encoding ablation axis
ORDINAL_MODES  = ['coral', 'corn']                    # ordinal-loss ablation axis
BLEND_W_ORD    = 0.7                                  # 0.7*ordinal + 0.3*regression, same as prior study

os.makedirs('training/model', exist_ok=True)

print(f'QUICK_MODE  : {QUICK_MODE}')
print(f'Image size  : {IMG_SIZE}')
print(f'K-folds     : {K_FOLDS}')
print(f'Max epochs  : {EPOCHS}  (ES patience {PATIENCE_ES}, LR patience {PATIENCE_LR})')
print(f'Encodings   : {ENCODINGS}')
print(f'Ordinal modes: {ORDINAL_MODES}')

## 2 · Image loading + multi-colorspace decoding

In [ ]:
def load_tif_rgb(path, size):
    img = tiff.imread(path)
    if img.ndim == 4: img = img[0]
    if img.ndim == 3 and img.shape[0] in (1,3,4):
        img = np.transpose(img, (1,2,0))
    if img.ndim == 2:        img = np.stack([img]*3, axis=-1)
    elif img.shape[-1] == 1: img = np.concatenate([img]*3, axis=-1)
    elif img.shape[-1] == 4: img = img[...,:3]
    if img.dtype != np.uint8:
        img = img.astype(np.float32)
        lo, hi = img.min(), img.max()
        img = (img - lo) / (hi - lo + 1e-8)
        img = (img * 255).astype(np.uint8)
    return cv2.resize(img, (size[1], size[0]), interpolation=cv2.INTER_AREA)


_CV2_FORWARD = {'LAB': cv2.COLOR_RGB2LAB, 'HSV': cv2.COLOR_RGB2HSV, 'YCrCb': cv2.COLOR_RGB2YCrCb}
_CV2_INVERSE = {'LAB': cv2.COLOR_LAB2RGB, 'HSV': cv2.COLOR_HSV2RGB, 'YCrCb': cv2.COLOR_YCrCb2RGB}

def convert_encoding(img_rgb_uint8, encoding):
    """RGB uint8 -> target encoding, uint8. 'RGB' is a passthrough copy."""
    if encoding == 'RGB':
        return img_rgb_uint8.copy()
    return cv2.cvtColor(img_rgb_uint8, _CV2_FORWARD[encoding])

def decode_to_rgb(img_uint8, encoding):
    """Inverse of convert_encoding -- for visualization/debugging only."""
    if encoding == 'RGB':
        return img_uint8.copy()
    return cv2.cvtColor(img_uint8, _CV2_INVERSE[encoding])

def normalize_channels_01(img_uint8, encoding):
    """cv2 uses DIFFERENT per-channel ranges for different uint8 encodings --
    most notably HSV's Hue channel is 0-179, not 0-255. Blindly dividing every
    encoding by 255 would leave HSV's H channel compressed into [0, 0.70]
    instead of [0, 1], feeding the CNN an inconsistent scale for that
    encoding only. Rescale each channel to its own true range instead, so
    every encoding is on a comparable ~[0,1] footing."""
    img = img_uint8.astype(np.float32)
    if encoding == 'HSV':
        out = img.copy()
        out[..., 0] = img[..., 0] / 179.0
        out[..., 1] = img[..., 1] / 255.0
        out[..., 2] = img[..., 2] / 255.0
        return out
    return img / 255.0   # RGB, LAB, YCrCb are all 0-255 per channel in OpenCV uint8


def collect_data(root, score_folders, img_size):
    """Walk each Score=NN (or plain NN) subfolder under root, load RGB nocc
    images, and pre-decode into every encoding in ENCODINGS."""
    rgb_list, label_list = [], []
    print(f"\nSearching root: {root}")
    if not os.path.exists(root):
        raise FileNotFoundError(f"NOCC_ROOT not found: {root}")

    top_items = sorted(os.listdir(root))
    print(f"  Contents ({len(top_items)} items): {top_items[:20]}")

    for folder_num in score_folders:
        candidates = [os.path.join(root, str(folder_num)),
                      os.path.join(root, f"Score={folder_num:02d}"),
                      os.path.join(root, f"Score={folder_num}"),
                      os.path.join(root, f"{folder_num:02d}")]
        folder_path = next((p for p in candidates if os.path.isdir(p)), None)
        if folder_path is None:
            print(f"  [Score {folder_num}] ⚠  No matching folder found")
            continue

        all_tifs = list(set(glob.glob(os.path.join(folder_path, "*.tif"))
                           + glob.glob(os.path.join(folder_path, "*.TIF"))
                           + glob.glob(os.path.join(folder_path, "*.tiff"))))
        nocc_tifs = [p for p in all_tifs if "nocc" in os.path.basename(p).lower()]
        chosen = nocc_tifs if nocc_tifs else all_tifs
        tag = "nocc" if nocc_tifs else "ALL (no nocc found -- using fallback)"
        print(f"  [Score {folder_num}] folder={os.path.basename(folder_path)} "
              f"| all_tifs={len(all_tifs)} | {tag}={len(chosen)}")

        for p in chosen:
            try:
                rgb = load_tif_rgb(p, img_size)
                rgb_list.append(rgb)
                label_list.append(folder_num - 1)
            except Exception as e:
                print(f"    ⚠ Failed {os.path.basename(p)}: {e}")

    if len(rgb_list) == 0:
        raise RuntimeError(f"No images loaded! Check NOCC_ROOT: {root}")

    print(f"\n  ✔ Total loaded: {len(rgb_list)} images across "
          f"{len(set(label_list))} score classes\n")
    return (np.array(rgb_list, dtype=np.uint8), np.array(label_list, dtype=np.int32))


X_rgb, y_raw = collect_data(NOCC_ROOT, SCORE_FOLDERS, IMG_SIZE)

print("Decoding into every encoding ...")
X_encoded = {'RGB': X_rgb}
for enc in ENCODINGS:
    if enc == 'RGB':
        continue
    X_encoded[enc] = np.array([convert_encoding(img, enc) for img in X_rgb], dtype=np.uint8)
    print(f"  {enc:6s}: {X_encoded[enc].shape}")

unique, counts = np.unique(y_raw, return_counts=True)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(unique + 1, counts, color='steelblue', edgecolor='black')
ax.set_xlabel('Score'); ax.set_ylabel('Count'); ax.set_title('Image Distribution per Score Class')
ax.set_xticks(unique + 1)
for u, c in zip(unique, counts):
    ax.text(u + 1, c + 0.3, str(c), ha='center', fontsize=9)
plt.tight_layout(); plt.savefig('training/class_distribution.png'); plt.show()

## 3 · Generic 54-dim colorspace feature extractor

Same recipe validated in the prior notebook (mean/median/skew/kurtosis/histogram per channel + spatial structure + cross-channel spread), but written generically over channels `c0,c1,c2` so it applies unchanged to any 3-channel uint8 image, whatever the colorspace.

In [ ]:
def extract_colorspace_features(img_uint8):
    """Returns a 54-dim float32 feature vector from ANY 3-channel uint8 image
    (RGB, LAB, HSV, YCrCb, ...). Channel-agnostic version of the improved
    LAB feature extractor from the prior ablation study -- same 54 active
    dims (no dead padding), just named generically (c0/c1/c2) so it applies
    to whichever encoding is currently active.

      c0,c1,c2 marginal mean + std        (3ch x 2)   =  6 dims
      c0,c1,c2 marginal median            (3ch x 1)   =  3 dims
      c0,c1,c2 marginal skewness          (3ch x 1)   =  3 dims
      c0,c1,c2 marginal excess kurtosis   (3ch x 1)   =  3 dims
      c0,c1,c2 8-bin histograms           (8 x 3ch)   = 24 dims
      center-vs-border mean delta         (3ch)       =  3 dims  (spatial)
      c1,c2 joint magnitude mean/std      sqrt(c1^2+c2^2)          =  2 dims
      c1,c2 joint angle circular mean/var atan2(c2,c1)             =  2 dims
      c1,c2 correlation coefficient                                =  1 dim
      c1,c2 interquartile range (IQR)                              =  2 dims
      quadrant means (4 spatial quadrants, c0 channel)             =  4 dims
      edge-region c0 std (border ring, ~8% width)                  =  1 dim
      -----------------------------------------------------------------
      total                                                        = 54 dims
    """
    h, w = img_uint8.shape[:2]
    c0 = img_uint8[:, :, 0].astype(np.float32)
    c1_raw = img_uint8[:, :, 1].astype(np.float32)
    c2_raw = img_uint8[:, :, 2].astype(np.float32)
    c1 = c1_raw - 128.0     # centered version, used for the joint-magnitude/angle/IQR block
    c2 = c2_raw - 128.0

    feats = []
    channels = (c0, c1_raw, c2_raw)

    for ch in channels:
        feats.append(ch.mean() / 255.0); feats.append(ch.std() / 255.0)
    for ch in channels:
        feats.append(float(np.median(ch)) / 255.0)
    for ch in channels:
        feats.append(float(np.nan_to_num(skew(ch.flatten()))))
    for ch in channels:
        feats.append(float(np.nan_to_num(kurtosis(ch.flatten()))))
    for ch in channels:
        hist, _ = np.histogram(ch, bins=8, range=(0, 256))
        hist = hist.astype(np.float32); hist /= (hist.sum() + 1e-8)
        feats.extend(hist.tolist())

    cy0, cy1 = h // 4, 3 * h // 4
    cx0, cx1 = w // 4, 3 * w // 4
    border_mask = np.ones((h, w), dtype=bool); border_mask[cy0:cy1, cx0:cx1] = False
    for ch in channels:
        feats.append((ch[cy0:cy1, cx0:cx1].mean() - ch[border_mask].mean()) / 255.0)

    magnitude = np.sqrt(c1**2 + c2**2)
    feats.append(magnitude.mean() / 180.0); feats.append(magnitude.std() / 180.0)

    angle = np.arctan2(c2, c1)
    sin_m, cos_m = np.sin(angle).mean(), np.cos(angle).mean()
    circ_mean = np.arctan2(sin_m, cos_m)
    circ_var  = 1.0 - np.sqrt(sin_m**2 + cos_m**2)
    feats.append((circ_mean + np.pi) / (2*np.pi)); feats.append(circ_var)

    if c1.std() > 1e-6 and c2.std() > 1e-6:
        corr = np.corrcoef(c1.flatten(), c2.flatten())[0, 1]
    else:
        corr = 0.0
    feats.append(float(np.nan_to_num(corr)))

    p75_1, p25_1 = np.percentile(c1, [75, 25]); p75_2, p25_2 = np.percentile(c2, [75, 25])
    feats.append(float(p75_1 - p25_1) / 180.0); feats.append(float(p75_2 - p25_2) / 180.0)

    hh, hw = h // 2, w // 2
    for q in (c0[:hh, :hw], c0[:hh, hw:], c0[hh:, :hw], c0[hh:, hw:]):
        feats.append(q.mean() / 255.0)

    ring_w = max(int(min(h, w) * 0.08), 2)
    ring_mask = np.zeros((h, w), dtype=bool)
    ring_mask[:ring_w, :] = True; ring_mask[-ring_w:, :] = True
    ring_mask[:, :ring_w] = True; ring_mask[:, -ring_w:] = True
    feats.append(c0[ring_mask].std() / 255.0)

    feats = np.array(feats, dtype=np.float32)
    if len(feats) < FEAT_DIM:
        feats = np.concatenate([feats, np.zeros(FEAT_DIM - len(feats), dtype=np.float32)])
    return feats[:FEAT_DIM]


print("Computing 54-dim features for every encoding ...")
feats_by_encoding = {}
for enc in ENCODINGS:
    feats_by_encoding[enc] = np.array(
        [extract_colorspace_features(img) for img in X_encoded[enc]], dtype=np.float32)
    print(f"  {enc:6s}: {feats_by_encoding[enc].shape}")

## 4 · Augmentation (generic over encodings)

Geometric augmentations (flip/rotate/zoom) are applied on the RGB image and then re-decoded into the target encoding, so they're geometrically correct regardless of colorspace. Photometric augmentations (blur/noise) are applied directly on the already-encoded image.

In [ ]:
def aug_flip(rgb, v):
    return cv2.flip(rgb, [1, 0, -1][v])

def aug_rotate(rgb, v):
    angles = [15, -15, 30]
    h, w = rgb.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angles[v], 1.0)
    return cv2.warpAffine(rgb, M, (w, h), borderMode=cv2.BORDER_REFLECT)

def aug_zoom(rgb, v):
    pcts = [0.10, 0.20, 0.30]; f = pcts[v]
    h, w = rgb.shape[:2]; dh, dw = int(h*f/2), int(w*f/2)
    return cv2.resize(rgb[dh:h-dh, dw:w-dw], (w, h), interpolation=cv2.INTER_LINEAR)

def aug_blur(img, v):
    return cv2.GaussianBlur(img, (3, 3), [0.5, 1.0, 1.5][v])

def aug_noise(img, v):
    stds = [3, 6, 9]
    n = np.random.normal(0, stds[v], img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) + n, 0, 255).astype(np.uint8)

AUG_CATALOGUE = [
    ('Flip',     aug_flip,   'rgb', 3),
    ('Rotation', aug_rotate, 'rgb', 3),
    ('Zoom',     aug_zoom,   'rgb', 3),
    ('Blur',     aug_blur,   'enc', 3),
    ('Noise',    aug_noise,  'enc', 3),
]
print(f'Augmentation factor : x{1 + sum(n for *_, n in AUG_CATALOGUE)}')


def augment_fold(train_idx, encoding):
    """Builds the augmented training set for one fold, in the requested
    encoding, recomputing color features for every augmented sample."""
    X_enc = X_encoded[encoding]
    enc_list, feat_list, y_list = [], [], []
    for i in train_idx:
        enc_list.append(X_enc[i]); feat_list.append(feats_by_encoding[encoding][i]); y_list.append(y_raw[i])
    for aug_name, aug_fn, domain, n_var in AUG_CATALOGUE:
        for v in range(n_var):
            for i in train_idx:
                if domain == 'rgb':
                    a_enc = convert_encoding(aug_fn(X_rgb[i], v), encoding)
                else:
                    a_enc = aug_fn(X_enc[i], v)
                enc_list.append(a_enc)
                feat_list.append(extract_colorspace_features(a_enc))
                y_list.append(y_raw[i])
    return (np.array(enc_list, dtype=np.uint8),
            np.array(feat_list, dtype=np.float32),
            np.array(y_list, dtype=np.int32))

print("Augmentation helpers defined ✔")

## 5 · Dataset (encoding-agnostic)

In [ ]:
class ZucchiniDataset(Dataset):
    """Encoded image (any colorspace) + optional handcrafted features,
    regression target = score 1..9. Channel normalization is encoding-aware
    (see normalize_channels_01) so e.g. HSV's 0-179 Hue channel isn't silently
    squashed by a blind /255."""
    def __init__(self, enc_imgs, feats, labels, encoding, augment=False, use_feats=True):
        normed = np.stack([normalize_channels_01(img, encoding) for img in enc_imgs])
        self.imgs   = torch.from_numpy(normed.astype(np.float32)).permute(0, 3, 1, 2)
        self.feats  = torch.from_numpy(feats.astype(np.float32)) if use_feats else None
        self.scores = torch.from_numpy(IDX_TO_SCORE[labels]).unsqueeze(1)
        self.labels = labels
        self.augment = augment
        self.use_feats = use_feats

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        img = self.imgs[idx].clone()
        score = self.scores[idx]
        if self.augment and random.random() < 0.3:
            if random.random() < 0.5: img = img.flip(2)
            if random.random() < 0.5: img = img.flip(1)
        feat = self.feats[idx] if self.use_feats else torch.zeros(FEAT_DIM)
        return img, feat, score, self.labels[idx]

print("ZucchiniDataset defined ✔")

## 6 · Stratified 5-fold splits (shared across every experiment)

In [ ]:
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_splits = list(skf.split(np.zeros(len(y_raw)), y_raw))
for i, (tr, va) in enumerate(fold_splits):
    print(f'  Fold {i+1}: train={len(tr)}  val={len(va)}')
print("Fold splits ready ✔ (shared by every experiment below, for fair comparison)")

## 7 · CNN backbone — 6 conv blocks

Deeper than the 4-block backbone used in the prior ablation study (which was `32->64->128->256` over 4 double-conv blocks). This one adds two more blocks and stops pooling on the last block so spatial dims don't vanish: `128 -> 64 -> 32 -> 16 -> 8 -> 4 -> 4`, GAP to a 384-dim embedding.

In [ ]:
class ConvBlock(nn.Module):
    """Double-conv block: Conv-BN-ReLU x2, optional MaxPool, then Dropout2d."""
    def __init__(self, in_ch, out_ch, dropout=0.20, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2))
        layers.append(nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class CNNBackbone6(nn.Module):
    """6 ConvBlocks: 128->64->32->16->8->4->4 (last block skips pooling to
    avoid over-shrinking spatial dims), then global-average-pool."""
    def __init__(self, in_ch=3):
        super().__init__()
        self.blocks = nn.Sequential(
            ConvBlock(in_ch, 32,  0.10),          # 128 -> 64
            ConvBlock(32,    64,  0.15),          # 64  -> 32
            ConvBlock(64,    128, 0.20),          # 32  -> 16
            ConvBlock(128,   192, 0.25),          # 16  -> 8
            ConvBlock(192,   256, 0.30),          # 8   -> 4
            ConvBlock(256,   384, 0.30, pool=False),  # 4 -> 4 (extra depth, no further shrink)
        )
        self.gap = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        return self.gap(self.blocks(x)).flatten(1)   # (B, 384)


CNN_DIM = 384
_n_params = sum(p.numel() for p in CNNBackbone6().parameters() if p.requires_grad)
print(f'CNNBackbone6: {_n_params:,} trainable params, output dim = {CNN_DIM}')

## 8 · Model heads — single-head baseline and dual-head (ordinal + regression)

One flexible class per head-type, each with a `use_color_feat` switch, so every combination in the grid below is a `model_cls(use_color_feat=...)` call rather than a bespoke class per composition.

In [ ]:
class ZucchiniSingleHeadNet(nn.Module):
    """Single regression head. use_color_feat=True fuses the 54-dim
    handcrafted feature vector; prior study found this consistently hurts,
    kept here only so §13 can re-validate that on the new backbone."""
    def __init__(self, use_color_feat=False):
        super().__init__()
        self.use_color_feat = use_color_feat
        self.backbone = CNNBackbone6()
        if use_color_feat:
            self.feat_mlp = nn.Sequential(nn.Linear(FEAT_DIM, 64), nn.ReLU(),
                                           nn.Linear(64, 32), nn.ReLU())
            fused_dim = CNN_DIM + 32
        else:
            self.feat_mlp = None
            fused_dim = CNN_DIM
        self.fc = nn.Sequential(
            nn.Linear(fused_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.35),
            nn.Linear(128, 1),
        )

    def forward(self, img, feat=None):
        cnn_out = self.backbone(img)
        fused = torch.cat([cnn_out, self.feat_mlp(feat)], dim=1) if self.use_color_feat else cnn_out
        return self.fc(fused)


class ZucchiniDualHeadNet(nn.Module):
    """Dual head (ordinal + regression), sharing a trunk after the backbone.
    use_color_feat=True fuses the 54-dim handcrafted feature vector (kept
    only for the confirmatory check in §16 -- the main grid runs with it
    off, per the prior study's finding)."""
    def __init__(self, use_color_feat=False):
        super().__init__()
        self.use_color_feat = use_color_feat
        self.backbone = CNNBackbone6()
        if use_color_feat:
            self.feat_mlp = nn.Sequential(nn.Linear(FEAT_DIM, 64), nn.ReLU(),
                                           nn.Linear(64, 32), nn.ReLU())
            fused_dim = CNN_DIM + 32
        else:
            self.feat_mlp = None
            fused_dim = CNN_DIM
        self.shared_fc = nn.Sequential(
            nn.Linear(fused_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.35),
        )
        self.ord_head = nn.Linear(128, NUM_ORD)
        self.reg_head = nn.Linear(128, 1)

    def forward(self, img, feat=None):
        cnn_out = self.backbone(img)
        fused = torch.cat([cnn_out, self.feat_mlp(feat)], dim=1) if self.use_color_feat else cnn_out
        shared = self.shared_fc(fused)
        return self.ord_head(shared), self.reg_head(shared)   # (B,NUM_ORD), (B,1)


for _cls, _kw in [(ZucchiniSingleHeadNet, dict(use_color_feat=False)),
                   (ZucchiniDualHeadNet,   dict(use_color_feat=False)),
                   (ZucchiniDualHeadNet,   dict(use_color_feat=True))]:
    _n = sum(p.numel() for p in _cls(**_kw).parameters() if p.requires_grad)
    print(f'{_cls.__name__}({_kw}) : {_n:,} trainable params')

## 9 · Ordinal decoding — CORAL (sum) vs CORN (filtered/conditional)

Both use the same `NUM_ORD` cumulative-threshold targets (`y_ord[:,k] = 1{score > k+1}`), but differ in *how the loss is computed* and *how a rank is decoded from probabilities*:

- **CORAL** (used in the prior notebook): each of the 8 thresholds is trained as an independent binary classifier via plain BCE on every sample. Decoding sums all 8 sigmoid probabilities: `rank = 1 + sum_k P(score > k)`.
- **CORN**: each threshold `k`'s BCE loss is only computed for samples that already satisfy threshold `k-1` — i.e. the training signal for each conditional binary task is **filtered down** to the subset it's actually defined for, rather than training all 8 thresholds unconditionally on every sample. Decoding uses the **cumulative product** of conditional probabilities: `rank = 1 + count_k[ prod_{j<=k} P(score>j | score>j-1) > 0.5 ]`, which is rank-consistent by construction.

In [ ]:
def decode_ordinal(ord_logits, mode='coral'):
    """Turns ordinal logits into a continuous 1..9 score estimate."""
    probs = torch.sigmoid(ord_logits)                      # (B, NUM_ORD)
    if mode == 'coral':
        return 1.0 + probs.sum(dim=1, keepdim=True)
    elif mode == 'corn':
        cum_probs = torch.cumprod(probs, dim=1)             # conditional cumulative probs
        return 1.0 + (cum_probs > 0.5).float().sum(dim=1, keepdim=True)
    raise ValueError(f'Unknown ordinal decode mode: {mode}')


def predict_reg_ord_blend(ord_logits, reg_out, ordinal_mode='coral', blend_w_ord=BLEND_W_ORD):
    """Splits a dual-head model's output into the three prediction sources
    used throughout this notebook: regression-only, ordinal-only, and the
    standard blend."""
    reg_score = reg_out.clamp(1.0, 9.0)
    ord_score = decode_ordinal(ord_logits, mode=ordinal_mode)
    blend     = blend_w_ord * ord_score + (1 - blend_w_ord) * reg_score
    return reg_score, ord_score, blend

print("Ordinal decode (CORAL/CORN) + reg/ord/blend split defined ✔")

## 10 · Loss functions

In [ ]:
def loss_mse(pred, target):
    return F.mse_loss(pred, target)

def loss_huber(reg_out, target, delta=1.0):
    return F.huber_loss(reg_out, target, delta=delta)

def loss_ranking(reg_out, target, margin=1.0):
    """Pairwise ranking loss: for pairs whose true scores differ by >=1,
    penalize the model if it doesn't rank them in the same order."""
    pred = reg_out.squeeze(1); t = target.squeeze(1)
    s_i, s_j = t.unsqueeze(1), t.unsqueeze(0)
    p_i, p_j = pred.unsqueeze(1), pred.unsqueeze(0)
    gap  = s_i - s_j
    mask = (gap.abs() >= 1.0).float()
    l    = torch.clamp(margin - gap.sign() * (p_i - p_j), min=0.0)
    return (mask * l).sum() / (mask.sum() + 1e-8)


def _ordinal_targets(y_score):
    thresholds = torch.arange(1, NUM_ORD + 1, dtype=torch.float32,
                               device=y_score.device).unsqueeze(0)   # (1, NUM_ORD)
    return (y_score.unsqueeze(1) > thresholds).float()               # (B, NUM_ORD)


def loss_ordinal_coral(y_score, ord_logits):
    """CORAL-style: independent BCE on every threshold, every sample."""
    y_ord = _ordinal_targets(y_score)
    return F.binary_cross_entropy_with_logits(ord_logits, y_ord)


def loss_ordinal_corn(y_score, ord_logits):
    """CORN: threshold k's BCE term is filtered to only the samples that
    already satisfy threshold k-1 (task 0 is always in scope). This trains
    genuinely conditional probabilities P(score>k | score>k-1) instead of
    independent per-threshold classifiers, which is what CORAL trains."""
    y_ord = _ordinal_targets(y_score)
    mask = torch.ones_like(y_ord)
    mask[:, 1:] = y_ord[:, :-1]     # task k in scope only if task k-1 was satisfied
    bce = F.binary_cross_entropy_with_logits(ord_logits, y_ord, reduction='none')
    return (bce * mask).sum() / mask.sum().clamp(min=1.0)


def loss_dual_mse(ord_logits, reg_out, target, ordinal_mode='coral'):
    """Regression MSE + ordinal loss (both heads must be trained)."""
    y_score = target.squeeze(1)
    l_reg = F.mse_loss(reg_out, target)
    l_ord = loss_ordinal_corn(y_score, ord_logits) if ordinal_mode == 'corn' \
            else loss_ordinal_coral(y_score, ord_logits)
    return l_reg + l_ord


def loss_composite(ord_logits, reg_out, target, ordinal_mode='coral'):
    """Composite loss: ordinal + huber (robust regression) + pairwise rank.
    Same vanilla weights (1.0/0.5/0.5) validated as best in the prior study."""
    y_score = target.squeeze(1)
    l_ord  = loss_ordinal_corn(y_score, ord_logits) if ordinal_mode == 'corn' \
             else loss_ordinal_coral(y_score, ord_logits)
    l_hub  = loss_huber(reg_out, target)
    l_rank = loss_ranking(reg_out, target)
    return 1.0 * l_ord + 0.5 * l_hub + 0.5 * l_rank


print("Loss functions defined ✔  (loss_composite/loss_dual_mse take ordinal_mode='coral'|'corn')")

## 11 · Metrics (R2, RMSE, MAE, rho, tau, QWK, Acc@0/1)

In [ ]:
def within_acc(preds, targets, gap):
    return float(np.mean(np.abs(np.round(preds) - targets) <= gap))

def compute_metrics(preds, targets):
    preds   = np.array(preds, dtype=np.float32).flatten()
    targets = np.array(targets, dtype=np.float32).flatten()
    r2   = r2_score(targets, preds)
    rmse = float(np.sqrt(mean_squared_error(targets, preds)))
    mae  = float(mean_absolute_error(targets, preds))
    rho, _ = spearmanr(targets, preds)
    tau, _ = kendalltau(targets, preds)
    acc0 = within_acc(preds, targets, 0)
    acc1 = within_acc(preds, targets, 1)
    p_int = np.clip(np.round(preds),   1, NUM_CLASSES).astype(int)
    t_int = np.clip(np.round(targets), 1, NUM_CLASSES).astype(int)
    qwk = float(cohen_kappa_score(t_int, p_int, weights='quadratic'))
    return dict(r2=r2, rmse=rmse, mae=mae, rho=float(rho), tau=float(tau),
                qwk=qwk, acc0=acc0, acc1=acc1)

print("compute_metrics defined ✔  (r2, rmse, mae, rho, tau, qwk, acc0, acc1)")

## 12 · Generic training engine

One engine covers every combination in the grid: any encoding, any head type, color-feat on/off, any loss function, any ordinal decode mode. This avoids the ad hoc per-model-ID branching the earlier notebook accumulated -- every experiment below is just a different set of arguments to `run_experiment()`.

In [ ]:
def make_loaders(enc_tr, feat_tr, y_tr, enc_va, feat_va, y_va, encoding, batch_size):
    ds_tr = ZucchiniDataset(enc_tr, feat_tr, y_tr, encoding, augment=True,  use_feats=True)
    ds_va = ZucchiniDataset(enc_va, feat_va, y_va, encoding, augment=False, use_feats=True)
    tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True,  num_workers=0)
    va = DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=0)
    return tr, va


def train_one_fold(model_cls, model_kwargs, run_id, fold_idx, tr_idx, va_idx,
                    encoding, head_type, loss_fn, ordinal_mode='coral', keep_mask=None):
    """
    head_type : 'single' or 'dual'
    loss_fn   : single -> loss_fn(pred, target)
                dual   -> loss_fn(ord_logits, reg_out, target)   [ordinal_mode baked in via a lambda/partial]
    keep_mask : optional boolean array over the full dataset (len == len(y_raw)).
                Statistical-filtering support: if given, samples with
                keep_mask[i]==False are dropped from the TRAINING split only
                -- the validation split is always the full, unfiltered set,
                so OOF metrics stay comparable to every unfiltered run.
    Returns per-sample OOF reg/ord/blend arrays for this fold's val split,
    plus training history and the checkpoint path. For 'single' head, ord and
    blend are just the reg prediction (there's nothing else to report).
    """
    if keep_mask is not None:
        tr_idx = tr_idx[keep_mask[tr_idx]]
    lab_tr_f, feat_tr_f, y_tr_f = augment_fold(tr_idx, encoding)
    enc_val_f = X_encoded[encoding][va_idx]
    feat_val_f = feats_by_encoding[encoding][va_idx]
    y_val_f = y_raw[va_idx]

    tr_loader, va_loader = make_loaders(lab_tr_f, feat_tr_f, y_tr_f,
                                         enc_val_f, feat_val_f, y_val_f,
                                         encoding, BATCH_SIZE)

    model = model_cls(**model_kwargs).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=LR_INIT)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=LR_FACTOR, patience=PATIENCE_LR, min_lr=LR_MIN)

    use_feat = model_kwargs.get('use_color_feat', False)
    best_val, best_state, best_epoch, no_improve = float('inf'), None, 0, 0
    history = []

    for epoch in range(EPOCHS):
        model.train()
        for img, feat, score, _ in tr_loader:
            img, score = img.to(device), score.to(device)
            feat = feat.to(device) if use_feat else None
            opt.zero_grad()
            if head_type == 'single':
                out = model(img, feat)
                loss = loss_fn(out, score)
            else:
                ord_l, reg_o = model(img, feat)
                loss = loss_fn(ord_l, reg_o, score)
            loss.backward()
            opt.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for img, feat, score, _ in va_loader:
                img, score = img.to(device), score.to(device)
                feat = feat.to(device) if use_feat else None
                if head_type == 'single':
                    vloss = loss_fn(model(img, feat), score)
                else:
                    ord_l, reg_o = model(img, feat)
                    vloss = loss_fn(ord_l, reg_o, score)
                val_losses.append(vloss.item())
        val_loss = float(np.mean(val_losses))
        history.append(val_loss)
        sched.step(val_loss)

        if val_loss < best_val - 1e-5:
            best_val, best_epoch, no_improve = val_loss, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
        if no_improve >= PATIENCE_ES:
            break

    model.load_state_dict(best_state)
    ckpt_path = f'training/model/best_{run_id}_fold{fold_idx+1}.pt'
    torch.save(model.state_dict(), ckpt_path)

    model.eval()
    reg_l, ord_l_, blend_l = [], [], []
    with torch.no_grad():
        for img, feat, score, _ in va_loader:
            img = img.to(device)
            feat = feat.to(device) if use_feat else None
            if head_type == 'single':
                reg_score = model(img, feat).clamp(1.0, 9.0)
                reg_l.append(reg_score.cpu().numpy().flatten())
                ord_l_.append(reg_score.cpu().numpy().flatten())
                blend_l.append(reg_score.cpu().numpy().flatten())
            else:
                ord_logits, reg_out = model(img, feat)
                reg_score, ord_score, blend = predict_reg_ord_blend(ord_logits, reg_out, ordinal_mode)
                reg_l.append(reg_score.cpu().numpy().flatten())
                ord_l_.append(ord_score.cpu().numpy().flatten())
                blend_l.append(blend.cpu().numpy().flatten())

    return dict(reg=np.concatenate(reg_l), ordv=np.concatenate(ord_l_),
                blend=np.concatenate(blend_l), best_epoch=best_epoch,
                n_epochs=len(history), ckpt=ckpt_path)


def run_experiment(run_id, model_cls, model_kwargs, encoding, head_type, loss_fn,
                    ordinal_mode='coral', keep_mask=None, verbose=True):
    """Runs all K_FOLDS, collecting OOF reg/ord/blend predictions + per-fold
    and OOF metrics for each of the three prediction sources. keep_mask (see
    train_one_fold) applies statistical-filtering to the training split only."""
    n = len(y_raw)
    reg_all   = np.full(n, np.nan, dtype=np.float32)
    ord_all   = np.full(n, np.nan, dtype=np.float32)
    blend_all = np.full(n, np.nan, dtype=np.float32)
    fold_metrics = {'reg': [], 'ord': [], 'blend': []}

    t0 = time.time()
    for fold_idx, (tr_idx, va_idx) in enumerate(fold_splits):
        out = train_one_fold(model_cls, model_kwargs, run_id, fold_idx, tr_idx, va_idx,
                              encoding, head_type, loss_fn, ordinal_mode, keep_mask=keep_mask)
        reg_all[va_idx], ord_all[va_idx], blend_all[va_idx] = out['reg'], out['ordv'], out['blend']
        true_fold = IDX_TO_SCORE[y_raw[va_idx]]
        fold_metrics['reg'].append(compute_metrics(out['reg'], true_fold))
        fold_metrics['ord'].append(compute_metrics(out['ordv'], true_fold))
        fold_metrics['blend'].append(compute_metrics(out['blend'], true_fold))
        if verbose:
            print(f'    [{run_id}] fold {fold_idx+1}/{K_FOLDS} done '
                  f'(best epoch {out["best_epoch"]+1}/{out["n_epochs"]})')

    true_all = IDX_TO_SCORE[y_raw]
    oof_metrics = {
        'reg':   compute_metrics(reg_all,   true_all),
        'ord':   compute_metrics(ord_all,   true_all),
        'blend': compute_metrics(blend_all, true_all),
    }
    elapsed = (time.time() - t0) / 60
    if verbose:
        print(f'  ✔ {run_id} done in {elapsed:.1f} min | '
              f'blend OOF R2={oof_metrics["blend"]["r2"]:.4f}  '
              f'QWK={oof_metrics["blend"]["qwk"]:.4f}')
    return dict(reg=reg_all, ordv=ord_all, blend=blend_all,
                fold_metrics=fold_metrics, oof_metrics=oof_metrics, minutes=elapsed)

print("run_experiment() ready ✔ -- one call per configuration in the grid below")

## 13 · Re-validation on the 6-block backbone (LAB encoding)

Before spending compute on the new colorspace/CORN grid, confirm the prior study's core findings still hold on this deeper backbone: does the color-feature branch still hurt, and does dual head + composite loss still win? All 4 runs use LAB (the encoding the prior study used) and the CORAL ordinal decode (also matching the prior study), so this is a clean like-for-like check -- only the backbone has changed.

In [ ]:
EXP13_RUNS = {
    'baseline':        dict(model_cls=ZucchiniSingleHeadNet,
                             model_kwargs=dict(use_color_feat=False),
                             head_type='single', loss_fn=loss_mse,
                             label='Baseline (single head, no color, MSE)'),
    'color_feat':      dict(model_cls=ZucchiniSingleHeadNet,
                             model_kwargs=dict(use_color_feat=True),
                             head_type='single', loss_fn=loss_mse,
                             label='+ Color Feats'),
    'dual_head':       dict(model_cls=ZucchiniDualHeadNet,
                             model_kwargs=dict(use_color_feat=False),
                             head_type='dual',
                             loss_fn=lambda o,r,t: loss_dual_mse(o,r,t,'coral'),
                             label='+ Dual Head (no color)'),
    'dual_composite':  dict(model_cls=ZucchiniDualHeadNet,
                             model_kwargs=dict(use_color_feat=False),
                             head_type='dual',
                             loss_fn=lambda o,r,t: loss_composite(o,r,t,'coral'),
                             label='+ Dual Head + Composite Loss (no color)'),
}

EXP13_RESULTS = {}
for run_id, cfg in EXP13_RUNS.items():
    print(f'\n=== §13 -- {cfg["label"]} ===')
    EXP13_RESULTS[run_id] = run_experiment(
        f'e13_{run_id}', cfg['model_cls'], cfg['model_kwargs'],
        encoding='LAB', head_type=cfg['head_type'], loss_fn=cfg['loss_fn'],
        ordinal_mode='coral')

In [ ]:
rows = []
for run_id, cfg in EXP13_RUNS.items():
    res = EXP13_RESULTS[run_id]
    m  = res['oof_metrics']['blend']    # for single-head runs, blend==reg (see run_experiment)
    fm = pd.DataFrame(res['fold_metrics']['blend'])
    rows.append({
        'Arm': cfg['label'],
        'R2 (OOF)': f'{m["r2"]:.4f}', 'R2 (CV mean+-std)': f'{fm["r2"].mean():.4f}+-{fm["r2"].std():.4f}',
        'RMSE (OOF)': f'{m["rmse"]:.4f}', 'MAE (OOF)': f'{m["mae"]:.4f}',
        'QWK (OOF)': f'{m["qwk"]:.4f}', 'Acc@1 (OOF %)': f'{m["acc1"]*100:.1f}',
    })
df_e13 = pd.DataFrame(rows)
print('='*110); print('  §13 RE-VALIDATION -- 6-block backbone, LAB encoding'.center(110)); print('='*110)
print(df_e13.to_string(index=False)); print('='*110)
df_e13.to_csv('training/exp13_revalidation.csv', index=False)
print("Saved -> training/exp13_revalidation.csv")

## 14 · Main grid — Image Encoding × Ordinal Loss (CORAL vs CORN)

Fixed at the architecture §13 (re-)confirmed best: **dual head + composite loss, no color features**. Varies:

- **encoding**: RGB, LAB, HSV, YCrCb
- **ordinal_mode**: `coral` (independent per-threshold BCE) vs `corn` (conditional/filtered per-threshold BCE)

= 8 configurations x 5 folds = 40 fold-trainings. This is the expensive cell in the notebook -- see the `QUICK_MODE` toggle in §1 if you want a fast smoke-test pass first.

In [ ]:
EXP14_RESULTS = {}
for encoding in ENCODINGS:
    for ordinal_mode in ORDINAL_MODES:
        run_id = f'e14_{encoding}_{ordinal_mode}'
        label  = f'{encoding} + {ordinal_mode.upper()}'
        print(f'\n=== §14 -- {label} ===')
        loss_fn = lambda o, r, t, _m=ordinal_mode: loss_composite(o, r, t, _m)
        EXP14_RESULTS[run_id] = dict(
            label=label, encoding=encoding, ordinal_mode=ordinal_mode,
            result=run_experiment(run_id, ZucchiniDualHeadNet, dict(use_color_feat=False),
                                   encoding=encoding, head_type='dual', loss_fn=loss_fn,
                                   ordinal_mode=ordinal_mode))

In [ ]:
METRIC_COLS  = ['r2', 'rmse', 'mae', 'rho', 'tau', 'qwk', 'acc0', 'acc1']
METRIC_NAMES = {'r2':'R2','rmse':'RMSE','mae':'MAE','rho':'Spearman rho','tau':'Kendall tau',
                'qwk':'QWK','acc0':'Acc@0 (%)','acc1':'Acc@1 (%)'}
PCT_METRICS  = {'acc0', 'acc1'}
SOURCE_LABEL = {'reg':'Regression only', 'ord':'Ordinal only', 'blend':'Blend (0.7*ord+0.3*reg)'}

rows = []
for run_id, cfg in EXP14_RESULTS.items():
    res = cfg['result']
    for src in ('reg', 'ord', 'blend'):
        m  = res['oof_metrics'][src]
        fm = pd.DataFrame(res['fold_metrics'][src])
        row = {'Encoding x Ordinal Loss': cfg['label'], 'Prediction source': SOURCE_LABEL[src]}
        for mc in METRIC_COLS:
            scale = 100 if mc in PCT_METRICS else 1
            fmt   = '.1f' if mc in PCT_METRICS else '.4f'
            row[f'{METRIC_NAMES[mc]} (OOF)']          = f'{m[mc]*scale:{fmt}}'
            row[f'{METRIC_NAMES[mc]} (CV mean+-std)'] = f'{fm[mc].mean()*scale:{fmt}}+-{fm[mc].std()*scale:{fmt}}'
        rows.append(row)

df_e14 = pd.DataFrame(rows)
print('='*145); print('  §14 FULL GRID -- Image Encoding x Ordinal Loss, per-head OOF + 5-fold CV'.center(145)); print('='*145)
print(df_e14.to_string(index=False)); print('='*145)
df_e14.to_csv('training/exp14_encoding_ordinal_grid.csv', index=False)
print("Saved -> training/exp14_encoding_ordinal_grid.csv")

### 15 · Pick the winning configuration

In [ ]:
_ranked = sorted(EXP14_RESULTS.items(),
                  key=lambda kv: kv[1]['result']['oof_metrics']['blend']['r2'],
                  reverse=True)

print('Ranked by blended OOF R2 (best first):')
for run_id, cfg in _ranked:
    m = cfg['result']['oof_metrics']['blend']
    print(f'  {cfg["label"]:<18}  R2={m["r2"]:.4f}  RMSE={m["rmse"]:.4f}  '
          f'MAE={m["mae"]:.4f}  QWK={m["qwk"]:.4f}  Acc@1={m["acc1"]*100:.1f}%')

WINNING_RUN_ID       = _ranked[0][0]
WINNING_ENCODING     = EXP14_RESULTS[WINNING_RUN_ID]['encoding']
WINNING_ORDINAL_MODE = EXP14_RESULTS[WINNING_RUN_ID]['ordinal_mode']

print(f'\n🏆 Winning configuration: encoding={WINNING_ENCODING}, '
      f'ordinal_mode={WINNING_ORDINAL_MODE}  '
      f'(blend OOF R2={EXP14_RESULTS[WINNING_RUN_ID]["result"]["oof_metrics"]["blend"]["r2"]:.4f})')

## 16 · Confirmatory check — does color-feat still hurt at the winning encoding?

A final, targeted check rather than re-running the full factorial again: color-feat on vs off, at the winning encoding + ordinal mode, dual head + composite loss. If the prior finding transfers, `color_off` should beat `color_on`.

In [ ]:
EXP16_RESULTS = {}
for use_color in (False, True):
    run_id = f'e16_{"color_on" if use_color else "color_off"}'
    label  = f'{"+ Color Feats" if use_color else "No Color Feats"} ({WINNING_ENCODING}+{WINNING_ORDINAL_MODE.upper()})'
    print(f'\n=== §16 -- {label} ===')
    loss_fn = lambda o, r, t, _m=WINNING_ORDINAL_MODE: loss_composite(o, r, t, _m)
    EXP16_RESULTS[run_id] = dict(
        label=label, use_color=use_color,
        result=run_experiment(run_id, ZucchiniDualHeadNet, dict(use_color_feat=use_color),
                               encoding=WINNING_ENCODING, head_type='dual', loss_fn=loss_fn,
                               ordinal_mode=WINNING_ORDINAL_MODE))

print('\nColor-feature check at the winning config:')
for run_id, cfg in EXP16_RESULTS.items():
    m = cfg['result']['oof_metrics']['blend']
    print(f'  {cfg["label"]:<45}  R2={m["r2"]:.4f}  MAE={m["mae"]:.4f}  QWK={m["qwk"]:.4f}')

_off_r2 = EXP16_RESULTS['e16_color_off']['result']['oof_metrics']['blend']['r2']
_on_r2  = EXP16_RESULTS['e16_color_on']['result']['oof_metrics']['blend']['r2']

# True only if color feats actually, measurably helped here (>0.002 R2 margin,
# to avoid flipping the decision on noise) -- otherwise keep them off, per the
# prior study's consistent finding.
USE_COLOR_FEAT_FINAL = (_on_r2 - _off_r2) > 0.002

print(f'\nWith-color R2={_on_r2:.4f}  vs  No-color R2={_off_r2:.4f}')
if USE_COLOR_FEAT_FINAL:
    print('-> Color feats measurably HELPED here (unexpected vs the prior study) -- keeping them ON.')
else:
    print('-> Confirms the prior finding: color feats do not help -- final pipeline uses NO color feats.')
print(f'USE_COLOR_FEAT_FINAL = {USE_COLOR_FEAT_FINAL}')

## 17 · The Ultimate Pipeline — final assembled configuration

Combines every winning ingredient found across both notebooks:

- **6-conv-block CNN backbone** (§7)
- **Winning image encoding** (from §14/§15)
- **Dual head** (ordinal + regression), no color-feature branch unless §16 said otherwise
- **Composite loss** (huber + rank + ordinal, ordinal term using the winning CORAL/CORN mode from §14/§15)
- **reg / ord / blend** reporting, so the deployed model exposes all three scores, not just the blend

This retrains the winning configuration fresh (5-fold CV) so the final checkpoints in `training/model/` correspond exactly to `FINAL_*` settings, even if §14 already covered this exact combination.

In [ ]:
FINAL_ENCODING     = WINNING_ENCODING
FINAL_ORDINAL_MODE = WINNING_ORDINAL_MODE
FINAL_USE_COLOR    = USE_COLOR_FEAT_FINAL

print('Ultimate pipeline configuration:')
print(f'  Backbone       : CNNBackbone6 (6 conv blocks, {CNN_DIM}-dim embedding)')
print(f'  Image encoding : {FINAL_ENCODING}')
print(f'  Head type      : dual (ordinal + regression)')
print(f'  Color features : {"ON" if FINAL_USE_COLOR else "OFF"}')
print(f'  Ordinal mode   : {FINAL_ORDINAL_MODE.upper()}')
print(f'  Loss           : composite (1.0*ordinal[{FINAL_ORDINAL_MODE}] + 0.5*huber + 0.5*rank)')
print(f'  Blend          : {BLEND_W_ORD}*ordinal + {1-BLEND_W_ORD:.1f}*regression')

_final_loss_fn = lambda o, r, t, _m=FINAL_ORDINAL_MODE: loss_composite(o, r, t, _m)

FINAL_RESULT = run_experiment(
    'FINAL', ZucchiniDualHeadNet, dict(use_color_feat=FINAL_USE_COLOR),
    encoding=FINAL_ENCODING, head_type='dual', loss_fn=_final_loss_fn,
    ordinal_mode=FINAL_ORDINAL_MODE)

### 17a · Final per-head diagnostic table (reg / ord / blend)

In [ ]:
rows = []
for src in ('reg', 'ord', 'blend'):
    m  = FINAL_RESULT['oof_metrics'][src]
    fm = pd.DataFrame(FINAL_RESULT['fold_metrics'][src])
    row = {'Prediction source': SOURCE_LABEL[src]}
    for mc in METRIC_COLS:
        scale = 100 if mc in PCT_METRICS else 1
        fmt   = '.1f' if mc in PCT_METRICS else '.4f'
        row[f'{METRIC_NAMES[mc]} (OOF)']          = f'{m[mc]*scale:{fmt}}'
        row[f'{METRIC_NAMES[mc]} (CV mean+-std)'] = f'{fm[mc].mean()*scale:{fmt}}+-{fm[mc].std()*scale:{fmt}}'
    rows.append(row)

df_final = pd.DataFrame(rows)
print('='*130); print('  ULTIMATE PIPELINE -- Final Model, reg/ord/blend, OOF + 5-fold CV'.center(130)); print('='*130)
print(df_final.to_string(index=False)); print('='*130)
df_final.to_csv('training/final_pipeline_results.csv', index=False)
print("Saved -> training/final_pipeline_results.csv")

_best_src = max(('reg','ord','blend'), key=lambda s: FINAL_RESULT['oof_metrics'][s]['r2'])
print(f'\nBest single prediction source: {SOURCE_LABEL[_best_src]} '
      f'(R2={FINAL_RESULT["oof_metrics"][_best_src]["r2"]:.4f}) vs '
      f'blend (R2={FINAL_RESULT["oof_metrics"]["blend"]["r2"]:.4f})')

## 18 · Deployment — `predict_quality()`

Loads all 5 fold checkpoints for the final configuration and averages their predictions (a simple 5-way ensemble), returning reg / ord / blend scores for a single new raw RGB image.

In [ ]:
def predict_quality(rgb_uint8_image):
    """rgb_uint8_image: HxWx3 uint8 RGB image (already cropped to the produce,
    any size -- it will be resized to IMG_SIZE). Returns a dict with reg/ord/
    blend scores, each averaged across the 5 fold checkpoints."""
    img = cv2.resize(rgb_uint8_image, (IMG_SIZE[1], IMG_SIZE[0]), interpolation=cv2.INTER_AREA)
    enc_img = convert_encoding(img, FINAL_ENCODING)
    feat    = extract_colorspace_features(enc_img) if FINAL_USE_COLOR else np.zeros(FEAT_DIM, dtype=np.float32)

    normed = normalize_channels_01(enc_img, FINAL_ENCODING)
    img_t  = torch.from_numpy(normed.astype(np.float32)).permute(2, 0, 1).unsqueeze(0).to(device)
    feat_t = torch.from_numpy(feat.astype(np.float32)).unsqueeze(0).to(device)

    reg_preds, ord_preds, blend_preds = [], [], []
    model = ZucchiniDualHeadNet(use_color_feat=FINAL_USE_COLOR).to(device)
    model.eval()
    with torch.no_grad():
        for fold_idx in range(K_FOLDS):
            ckpt = f'training/model/best_FINAL_fold{fold_idx+1}.pt'
            model.load_state_dict(torch.load(ckpt, map_location=device))
            ord_logits, reg_out = model(img_t, feat_t if FINAL_USE_COLOR else None)
            reg_score, ord_score, blend = predict_reg_ord_blend(ord_logits, reg_out, FINAL_ORDINAL_MODE)
            reg_preds.append(reg_score.item()); ord_preds.append(ord_score.item()); blend_preds.append(blend.item())

    return dict(
        reg_score=float(np.mean(reg_preds)),  reg_std=float(np.std(reg_preds)),
        ord_score=float(np.mean(ord_preds)),  ord_std=float(np.std(ord_preds)),
        blend_score=float(np.mean(blend_preds)), blend_std=float(np.std(blend_preds)),
        n_folds_ensembled=K_FOLDS,
    )

print("predict_quality() ready ✔ -- call it with a new HxWx3 uint8 RGB image")
print("Example: predict_quality(X_rgb[0])")

## 19 · Summary — the ultimate pipeline, and why

In [ ]:
bm = FINAL_RESULT["oof_metrics"]["blend"]

summary_lines = [
    "="*80,
    "ULTIMATE PIPELINE SUMMARY".center(80),
    "="*80,
    "",
    f"Architecture     : 6-conv-block CNN backbone ({CNN_DIM}-dim embedding)",
    f"                    -> shared FC(128) -> ordinal head (NUM_ORD={NUM_ORD}) + regression head",
    f"Image encoding    : {FINAL_ENCODING}   (chosen from RGB/LAB/HSV/YCrCb in Sec14/15)",
    f'Color features    : {"ON" if FINAL_USE_COLOR else "OFF"}   '
        f'(re-validated in Sec13/16; prior study found this hurts)',
    f"Ordinal loss      : {FINAL_ORDINAL_MODE.upper()}   (chosen from CORAL vs CORN in Sec14/15)",
    f"Full loss         : 1.0*ordinal[{FINAL_ORDINAL_MODE}] + 0.5*huber + 0.5*rank  (vanilla weights --",
    "                    every tuned-weight variant tried in the prior notebook underperformed these",
    "                    once honestly cross-validated, so they were not re-tuned here)",
    f"Scoring           : reg (regression head alone), ord (ordinal head alone),",
    f"                    blend = {BLEND_W_ORD}*ord + {1-BLEND_W_ORD:.1f}*reg  -- all three reported,",
    "                    since the prior study found the ordinal head often carries most of the",
    "                    signal and the blend mainly reduces MAE rather than raising the ceiling",
    f"Final OOF (blend) : R2={bm['r2']:.4f}  RMSE={bm['rmse']:.4f}  MAE={bm['mae']:.4f}  "
        f"QWK={bm['qwk']:.4f}  Acc@1={bm['acc1']*100:.1f}%",
    "Deployment        : predict_quality(rgb_image) -- 5-fold checkpoint ensemble,",
    "                    returns reg/ord/blend scores + their cross-fold std",
    "="*80,
]
print("\n".join(summary_lines))

## 20 · Frieze visualization — fixed for the real class count, bigger images

The frieze plots used elsewhere hardcoded 9 classes and squeezed every image
into a ~1-inch-tall strip. Two fixes:

- **Class count is now data-driven**, not hardcoded. This dataset's real
  score range is **1–8** (score 9 has zero samples), so the frieze now
  reads that off the data itself (`int(np.floor(values.max()))`) instead of
  assuming 9 — the transition markers and spare columns match whatever the
  data actually contains, for this dataset or any other.
- **Images are bigger**: each frieze cell is now ~1.8" wide × ~2.6" tall
  (tune via `img_w`/`img_h`), instead of the ~1"-tall strip before.

One reusable function replaces the three near-duplicated frieze cells from
the regression notebook.

In [ ]:
def plot_frieze(images_rgb01, values, title='', img_w=1.8, img_h=2.6,
                 fontsize_title=11, fontsize_marker=26):
    """
    images_rgb01 : (N, H, W, 3) array, RGB, already in [0,1] (NOT the
                   normalized-for-CNN [-1,1] tensors -- pass X_rgb-derived
                   images so the frieze is human-viewable regardless of
                   which encoding the model was actually trained on).
    values       : (N,) array used both to sort the frieze and as each
                   image's title (predicted or true score).
    Inserts a bold class-number marker column each time the floor of
    `values` crosses into a new integer class. n_classes is read from the
    data (max floor(value) present), not hardcoded.
    """
    values = np.asarray(values, dtype=np.float32)
    n_classes = int(np.floor(values.max()))
    sorted_idx  = np.argsort(values)
    sorted_imgs = images_rgb01[sorted_idx]
    sorted_vals = values[sorted_idx]

    n_cols = len(sorted_imgs) + n_classes
    fig, ax = plt.subplots(1, n_cols, figsize=(n_cols * img_w, img_h))

    class_int = 0
    for i, (img, val) in enumerate(zip(sorted_imgs, sorted_vals)):
        disp = np.clip(img, 0, 1)
        class_tmp = int(np.floor(val))
        if class_tmp > class_int:
            ax[i + class_int].axis('off')
            ax[i + class_int].text(0.5, 0.5, f'{class_tmp}', fontsize=fontsize_marker,
                                    fontweight='bold', ha='center', va='center')
            while class_int < class_tmp:
                class_int += 1
                ax[i + class_int].set_title(f'{val:.2f}', fontsize=fontsize_title)
                ax[i + class_int].imshow(disp)
                ax[i + class_int].axis('off')
        else:
            ax[i + class_int].set_title(f'{val:.2f}', fontsize=fontsize_title)
            ax[i + class_int].imshow(disp)
            ax[i + class_int].axis('off')
    while class_int < n_classes:
        class_int += 1
        ax[len(sorted_imgs) + class_int - 1].axis('off')
        ax[len(sorted_imgs) + class_int - 1].text(0.5, 0.5, f'{class_int}', fontsize=fontsize_marker,
                                                    fontweight='bold', ha='center', va='center')
    if title:
        fig.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


X_rgb01 = X_rgb.astype(np.float32) / 255.0   # human-viewable images, reused by every frieze below
TRUE_SCORES = IDX_TO_SCORE[y_raw]
print(f'plot_frieze() ready ✔  -- data spans {int(TRUE_SCORES.min())}-{int(TRUE_SCORES.max())} '
      f'({int(np.floor(TRUE_SCORES.max()))} classes present)')

plot_frieze(X_rgb01, TRUE_SCORES, title='Reference: True Labels')

## 21 · Ablation gallery — reg / ord / blend × filtering-or-not (CORAL vs CORN)

Directly visualizes the two things asked for: how the three prediction
sources (**regression-only, ordinal-only, blend**) order the images, and
whether **CORN's conditional filtering** ("filtering") changes that
ordering versus plain **CORAL** ("not filtering") -- at the winning
encoding found in §14/§15. Built entirely from the OOF predictions already
computed in §14 -- no retraining.

In [ ]:
for ordinal_mode in ORDINAL_MODES:   # ['coral', 'corn']
    run_id = f'e14_{WINNING_ENCODING}_{ordinal_mode}'
    res = EXP14_RESULTS[run_id]['result']
    tag = 'CORN (filtered)' if ordinal_mode == 'corn' else 'CORAL (not filtered)'
    for src, arr_key in (('Regression only', 'reg'), ('Ordinal only', 'ordv'), ('Blend', 'blend')):
        plot_frieze(X_rgb01, res[arr_key],
                    title=f'{WINNING_ENCODING} + {tag} -- {src}')

## 22 · Colorspace comparison gallery — blend predictions, all 4 encodings

Same blend prediction source, same ordinal mode (the winning one), varying
only the image encoding the CNN was trained on -- directly answers
"comparing all the colorspaces" visually, side by side with the frieze
from §21 above.

In [ ]:
for encoding in ENCODINGS:
    run_id = f'e14_{encoding}_{WINNING_ORDINAL_MODE}'
    res = EXP14_RESULTS[run_id]['result']
    plot_frieze(X_rgb01, res['blend'],
                title=f'{encoding} + {WINNING_ORDINAL_MODE.upper()} -- Blend')

## 23 · Statistical filtering — outlier removal as a real ablation axis

"Filtering" here means **statistical outlier removal**: dropping training
images whose handcrafted 54-dim feature vector (§3) is a statistical outlier
*relative to its own score class* -- e.g. a mislabeled sample, a lighting
glitch, or a genuinely atypical zucchini. This is applied to the **training
split only** in every fold; the validation split always stays the full,
unfiltered set, so filtered and unfiltered runs remain directly comparable
on the same held-out data.

**Method**: per class, per feature dimension, compute the **modified
z-score** (Iglewicz & Hoaglin: `0.6745*(x-median)/MAD`) -- a robust
alternative to a standard z-score that isn't itself thrown off by the
outliers it's trying to detect. Average the absolute modified z-score across
all 54 dimensions to get one outlier score per image; flag anything above
the threshold (3.5 is the standard default for this statistic).

**Caveat worth flagging**: some classes here have as few as 11 samples
(score=1). MAD-based statistics on that few points are noisy -- a couple of
genuinely unusual-but-valid images could get flagged just from small-sample
variance. Treat the filtering decision in §24 as a *hypothesis to test*, not
a foregone conclusion -- which is exactly why it's run as an ablation rather
than applied unconditionally.

In [ ]:
def compute_statistical_outlier_scores(features, labels, eps=1e-6):
    """Per-class modified z-score (Iglewicz & Hoaglin), averaged across all
    54 feature dimensions -- one scalar outlier score per sample."""
    n = len(labels)
    scores = np.zeros(n, dtype=np.float32)
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        X_c = features[idx]
        median = np.median(X_c, axis=0)
        mad = np.median(np.abs(X_c - median), axis=0) + eps
        mz = 0.6745 * (X_c - median) / mad
        scores[idx] = np.mean(np.abs(mz), axis=1)
    return scores


def apply_statistical_filtering(features, labels, threshold=3.5):
    """Boolean keep-mask over the full dataset: True = keep (not an outlier)."""
    scores = compute_statistical_outlier_scores(features, labels)
    return (scores <= threshold), scores


FILTER_THRESHOLD = 3.5
outlier_scores = compute_statistical_outlier_scores(feats_by_encoding[WINNING_ENCODING], y_raw)
KEEP_MASK = outlier_scores <= FILTER_THRESHOLD

print(f'Statistical filtering @ threshold={FILTER_THRESHOLD}:')
print(f'  Flagged as outliers : {(~KEEP_MASK).sum()} / {len(y_raw)} images')
for c in np.unique(y_raw):
    idx = np.where(y_raw == c)[0]
    n_flagged = (~KEEP_MASK[idx]).sum()
    print(f'    score {int(IDX_TO_SCORE[c])}: {n_flagged}/{len(idx)} flagged '
          f'(outlier score range {outlier_scores[idx].min():.2f}-{outlier_scores[idx].max():.2f})')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(outlier_scores, bins=20, color='steelblue', edgecolor='black', alpha=0.8)
ax.axvline(FILTER_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'threshold={FILTER_THRESHOLD}')
ax.set_xlabel('Outlier score (mean |modified z-score| across 54 features)')
ax.set_ylabel('Count'); ax.set_title('Statistical outlier score distribution')
ax.legend(); plt.tight_layout(); plt.show()

## 24 · Confirmatory check — is filtering actually beneficial?

Filtering on vs off, at the winning encoding/ordinal-mode, dual head +
composite loss, with §16's color-feat decision already applied. Same
"only adopt it if it clears a real margin" rule as §16 (>0.002 R²), since
throwing away real data is the more invasive choice and shouldn't win on
noise.

In [ ]:
EXP24_RESULTS = {}
_loss_fn_final = lambda o, r, t, _m=WINNING_ORDINAL_MODE: loss_composite(o, r, t, _m)
for use_filter in (False, True):
    run_id = f'e24_{"filtered" if use_filter else "unfiltered"}'
    label  = f'{"Filtered" if use_filter else "Unfiltered"} ({WINNING_ENCODING}+{WINNING_ORDINAL_MODE.upper()})'
    print(f'\n=== §24 -- {label} ===')
    EXP24_RESULTS[run_id] = dict(
        label=label, use_filter=use_filter,
        result=run_experiment(run_id, ZucchiniDualHeadNet, dict(use_color_feat=FINAL_USE_COLOR),
                               encoding=WINNING_ENCODING, head_type='dual', loss_fn=_loss_fn_final,
                               ordinal_mode=WINNING_ORDINAL_MODE,
                               keep_mask=KEEP_MASK if use_filter else None))

print('\nFiltering check:')
for run_id, cfg in EXP24_RESULTS.items():
    m = cfg['result']['oof_metrics']['blend']
    print(f'  {cfg["label"]:<40}  R2={m["r2"]:.4f}  RMSE={m["rmse"]:.4f}  MAE={m["mae"]:.4f}')

_filt_r2   = EXP24_RESULTS['e24_filtered']['result']['oof_metrics']['blend']['r2']
_unfilt_r2 = EXP24_RESULTS['e24_unfiltered']['result']['oof_metrics']['blend']['r2']
FINAL_USE_FILTERING = (_filt_r2 - _unfilt_r2) > 0.002

print(f'\nFiltered R2={_filt_r2:.4f}  vs  Unfiltered R2={_unfilt_r2:.4f}')
print('-> Filtering HELPS -- keeping it ON.' if FINAL_USE_FILTERING else
      '-> No clear benefit -- final pipeline uses NO statistical filtering.')
print(f'FINAL_USE_FILTERING = {FINAL_USE_FILTERING}')

## 25 · Loss-family ablation — is dual head + composite loss actually worth it?

At the now-finalized encoding, color-feat setting, and filtering setting,
compares 5 architecture/loss combinations head to head:

| Run | Head | Loss | Ordinal mode |
|---|---|---|---|
| `single_mse` | single | plain MSE | n/a -- the true baseline, no dual head at all |
| `dual_mse_coral` | dual | ordinal BCE + regression MSE | CORAL |
| `dual_mse_corn` | dual | ordinal BCE + regression MSE | CORN |
| `composite_coral` | dual | ordinal + huber + rank | CORAL |
| `composite_corn` | dual | ordinal + huber + rank | CORN |

This directly answers both questions at once: does adding the dual head
help at all (`single_mse` vs the other four), and does composite loss add
anything over plain `loss_dual_mse` once you're already committed to a dual
head (`dual_mse_*` vs `composite_*`) -- for both CORAL and CORN.

In [ ]:
EXP25_CONFIGS = {
    'single_mse': dict(
        model_cls=ZucchiniSingleHeadNet, head_type='single', ordinal_mode='coral',
        loss_fn=loss_mse, label='Single head, plain MSE (no dual head)'),
    'dual_mse_coral': dict(
        model_cls=ZucchiniDualHeadNet, head_type='dual', ordinal_mode='coral',
        loss_fn=lambda o,r,t: loss_dual_mse(o,r,t,'coral'),
        label='Dual head, loss_dual_mse, CORAL'),
    'dual_mse_corn': dict(
        model_cls=ZucchiniDualHeadNet, head_type='dual', ordinal_mode='corn',
        loss_fn=lambda o,r,t: loss_dual_mse(o,r,t,'corn'),
        label='Dual head, loss_dual_mse, CORN'),
    'composite_coral': dict(
        model_cls=ZucchiniDualHeadNet, head_type='dual', ordinal_mode='coral',
        loss_fn=lambda o,r,t: loss_composite(o,r,t,'coral'),
        label='Dual head, composite loss, CORAL'),
    'composite_corn': dict(
        model_cls=ZucchiniDualHeadNet, head_type='dual', ordinal_mode='corn',
        loss_fn=lambda o,r,t: loss_composite(o,r,t,'corn'),
        label='Dual head, composite loss, CORN'),
}

_keep_mask_final = KEEP_MASK if FINAL_USE_FILTERING else None
EXP25_RESULTS = {}
for run_id, cfg in EXP25_CONFIGS.items():
    print(f'\n=== §25 -- {cfg["label"]} ===')
    EXP25_RESULTS[run_id] = run_experiment(
        run_id, cfg['model_cls'], dict(use_color_feat=FINAL_USE_COLOR),
        encoding=WINNING_ENCODING, head_type=cfg['head_type'], loss_fn=cfg['loss_fn'],
        ordinal_mode=cfg['ordinal_mode'], keep_mask=_keep_mask_final)

In [ ]:
rows = []
for run_id, cfg in EXP25_CONFIGS.items():
    m  = EXP25_RESULTS[run_id]['oof_metrics']['blend']
    fm = pd.DataFrame(EXP25_RESULTS[run_id]['fold_metrics']['blend'])
    rows.append({
        'Configuration': cfg['label'],
        'R2 (OOF)': f'{m["r2"]:.4f}', 'R2 (CV mean+-std)': f'{fm["r2"].mean():.4f}+-{fm["r2"].std():.4f}',
        'RMSE (OOF)': f'{m["rmse"]:.4f}', 'RMSE (CV mean+-std)': f'{fm["rmse"].mean():.4f}+-{fm["rmse"].std():.4f}',
        'MAE (OOF)': f'{m["mae"]:.4f}', 'QWK (OOF)': f'{m["qwk"]:.4f}', 'Acc@1 (OOF %)': f'{m["acc1"]*100:.1f}',
    })
df_e25 = pd.DataFrame(rows)
print('='*140); print('  §25 LOSS-FAMILY ABLATION'.center(140)); print('='*140)
print(df_e25.to_string(index=False)); print('='*140)
df_e25.to_csv('training/exp25_loss_family_ablation.csv', index=False)

# ── Selection rule: primary R2, RMSE as tiebreaker when R2 is within noise ──
_ranked25 = sorted(EXP25_RESULTS.items(), key=lambda kv: kv[1]['oof_metrics']['blend']['r2'], reverse=True)
_top_r2   = _ranked25[0][1]['oof_metrics']['blend']['r2']
_close    = [(rid, res) for rid, res in _ranked25 if _top_r2 - res['oof_metrics']['blend']['r2'] <= 0.003]
_winner25 = min(_close, key=lambda kv: kv[1]['oof_metrics']['blend']['rmse'])[0] if len(_close) > 1 else _ranked25[0][0]

print(f'\nRanked by OOF blend R2:')
for rid, res in _ranked25:
    m = res['oof_metrics']['blend']
    tag = '  <- within 0.003 R2 of top, RMSE tiebreak applied' if (rid, res) in _close and len(_close) > 1 else ''
    print(f'  {EXP25_CONFIGS[rid]["label"]:<42} R2={m["r2"]:.4f}  RMSE={m["rmse"]:.4f}{tag}')

FINAL_LOSS_CONFIG = EXP25_CONFIGS[_winner25]
print(f'\n🏆 Winning loss/architecture: {FINAL_LOSS_CONFIG["label"]}')

## 26 · The true final pipeline

Supersedes §17's `FINAL_RESULT` with the configuration §23-25 actually
selected -- encoding, color-feat, **statistical filtering**, and
**architecture/loss** are now all chosen by the same rule (OOF R², RMSE as
tiebreaker), not assumed. Retrains fresh so the saved checkpoints match
these exact settings.

In [ ]:
print('True final pipeline configuration:')
print(f'  Backbone           : CNNBackbone6 (6 conv blocks, {CNN_DIM}-dim embedding)')
print(f'  Image encoding     : {WINNING_ENCODING}')
print(f'  Color features     : {"ON" if FINAL_USE_COLOR else "OFF"}')
print(f'  Statistical filter : {"ON" if FINAL_USE_FILTERING else "OFF"} '
      f'({"excludes" if FINAL_USE_FILTERING else "keeps"} '
      f'{(~KEEP_MASK).sum()} flagged training images)')
print(f'  Architecture/loss  : {FINAL_LOSS_CONFIG["label"]}')

FINAL_RESULT = run_experiment(
    'FINAL', FINAL_LOSS_CONFIG['model_cls'], dict(use_color_feat=FINAL_USE_COLOR),
    encoding=WINNING_ENCODING, head_type=FINAL_LOSS_CONFIG['head_type'],
    loss_fn=FINAL_LOSS_CONFIG['loss_fn'], ordinal_mode=FINAL_LOSS_CONFIG['ordinal_mode'],
    keep_mask=KEEP_MASK if FINAL_USE_FILTERING else None)

FINAL_ORDINAL_MODE = FINAL_LOSS_CONFIG['ordinal_mode']
FINAL_HEAD_TYPE     = FINAL_LOSS_CONFIG['head_type']

### 26a · Final metrics — reg / ord / blend, OOF + 5-fold CV

In [ ]:
_sources = ('reg', 'ord', 'blend') if FINAL_HEAD_TYPE == 'dual' else ('blend',)
rows = []
for src in _sources:
    m  = FINAL_RESULT['oof_metrics'][src]
    fm = pd.DataFrame(FINAL_RESULT['fold_metrics'][src])
    row = {'Prediction source': SOURCE_LABEL[src] if FINAL_HEAD_TYPE == 'dual' else 'Single head output'}
    for mc in METRIC_COLS:
        scale = 100 if mc in PCT_METRICS else 1
        fmt   = '.1f' if mc in PCT_METRICS else '.4f'
        row[f'{METRIC_NAMES[mc]} (OOF)']          = f'{m[mc]*scale:{fmt}}'
        row[f'{METRIC_NAMES[mc]} (CV mean+-std)'] = f'{fm[mc].mean()*scale:{fmt}}+-{fm[mc].std()*scale:{fmt}}'
    rows.append(row)

df_final = pd.DataFrame(rows)
print('='*130); print('  TRUE FINAL PIPELINE -- Metrics'.center(130)); print('='*130)
print(df_final.to_string(index=False)); print('='*130)
df_final.to_csv('training/final_pipeline_metrics.csv', index=False)
print("Saved -> training/final_pipeline_metrics.csv")

## 27 · Confusion matrix — final pipeline

In [ ]:
from sklearn.metrics import confusion_matrix

N_CLASSES_PRESENT = int(np.floor(IDX_TO_SCORE[y_raw].max()))
CLASS_LABELS = list(range(1, N_CLASSES_PRESENT + 1))

def plot_confusion(preds, true, title, class_labels=CLASS_LABELS):
    p_int = np.clip(np.round(preds), class_labels[0], class_labels[-1]).astype(int)
    t_int = np.clip(np.round(true),  class_labels[0], class_labels[-1]).astype(int)
    cm = confusion_matrix(t_int, p_int, labels=class_labels)
    cm_norm = cm.astype(np.float64) / cm.sum(axis=1, keepdims=True).clip(min=1)

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(class_labels))); ax.set_xticklabels(class_labels)
    ax.set_yticks(range(len(class_labels))); ax.set_yticklabels(class_labels)
    ax.set_xlabel('Predicted score', fontsize=12, fontweight='bold')
    ax.set_ylabel('True score', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold')
    for i in range(len(class_labels)):
        for j in range(len(class_labels)):
            if cm[i, j] > 0:
                color = 'white' if cm_norm[i, j] > 0.5 else 'black'
                ax.text(j, i, f'{cm[i,j]}', ha='center', va='center', color=color, fontsize=10)
    fig.colorbar(im, ax=ax, label='Row-normalized fraction')
    plt.tight_layout(); plt.show()
    return cm

_pred_final = FINAL_RESULT['blend'] if FINAL_HEAD_TYPE == 'dual' else FINAL_RESULT['reg']
cm_final = plot_confusion(_pred_final, TRUE_SCORES,
                           f'Final Pipeline -- {WINNING_ENCODING}, {FINAL_LOSS_CONFIG["label"]}')

## 28 · Frieze — final pipeline

In [ ]:
def plot_frieze_by_class(images_rgb01, values, true_labels, class_labels,
                          max_per_class=10, img_w=2.5, img_h=2.5,
                          fontsize_title=12, fontsize_row=15, title=''):
    """
    One row per score class, up to max_per_class example images per row,
    sorted left-to-right by ascending value (blend score). If a class has
    more than max_per_class images, evenly samples across the sorted order
    so the row still represents the full prediction spread within that
    class, not just the first N by index.
    """
    n_rows = len(class_labels)
    fig, axes = plt.subplots(n_rows, max_per_class,
                              figsize=(max_per_class * img_w, n_rows * img_h))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for row, c in enumerate(class_labels):
        idx = np.where(true_labels == c)[0]
        order = idx[np.argsort(values[idx])]
        if len(order) > max_per_class:
            pick = np.linspace(0, len(order) - 1, max_per_class).round().astype(int)
            order = order[pick]

        for col in range(max_per_class):
            ax = axes[row, col]
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            if col < len(order):
                img = np.clip(images_rgb01[order[col]], 0, 1)
                ax.imshow(img)
                ax.set_title(f'{values[order[col]]:.2f}', fontsize=fontsize_title)
            if col == 0:
                ax.set_ylabel(f'Score {c}', fontsize=fontsize_row, fontweight='bold')

    if title:
        fig.suptitle(title, fontsize=18, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


plot_frieze_by_class(X_rgb01, FINAL_RESULT['blend'], np.round(TRUE_SCORES).astype(int),
                      CLASS_LABELS, max_per_class=10,
                      title='FINAL PIPELINE -- Blend, by True Score Class')

## 29 · Per-class statistics + final summary

In [ ]:
rows = []
for c in CLASS_LABELS:
    idx = np.where(np.round(TRUE_SCORES) == c)[0]
    if len(idx) == 0:
        continue
    pred_c = _pred_final[idx]
    err_c  = np.abs(pred_c - TRUE_SCORES[idx])
    rows.append({
        'Score class': c,
        'N samples': len(idx),
        'Mean prediction': f'{pred_c.mean():.3f}',
        'Std prediction': f'{pred_c.std():.3f}',
        'Mean abs error': f'{err_c.mean():.3f}',
        'Acc@0 (%)': f'{(np.round(pred_c) == c).mean()*100:.1f}',
        'Acc@1 (%)': f'{(np.abs(np.round(pred_c) - c) <= 1).mean()*100:.1f}',
    })
df_perclass = pd.DataFrame(rows)
print('='*100); print('  PER-CLASS STATISTICS -- Final Pipeline'.center(100)); print('='*100)
print(df_perclass.to_string(index=False)); print('='*100)
df_perclass.to_csv('training/final_pipeline_per_class_stats.csv', index=False)
print("Saved -> training/final_pipeline_per_class_stats.csv")

In [ ]:
bm = FINAL_RESULT["oof_metrics"]["blend"] if FINAL_HEAD_TYPE == "dual" else FINAL_RESULT["oof_metrics"]["reg"]

summary_lines = [
    "="*80,
    "TRUE FINAL PIPELINE -- SUMMARY".center(80),
    "="*80,
    "",
    f"Backbone           : CNNBackbone6 (6 conv blocks, {CNN_DIM}-dim embedding)",
    f"Image encoding     : {WINNING_ENCODING}   (chosen in Sec14/15 across RGB/LAB/HSV/YCrCb)",
    f'Color features     : {"ON" if FINAL_USE_COLOR else "OFF"}   (Sec16 confirmatory check)',
    f'Statistical filter : {"ON" if FINAL_USE_FILTERING else "OFF"}   (Sec23/24; '
        f'{(~KEEP_MASK).sum()} images flagged as outliers at threshold={FILTER_THRESHOLD})',
    f"Architecture/loss  : {FINAL_LOSS_CONFIG['label']}   (Sec25 5-way ablation)",
    f"Selection rule     : OOF R2 primary, RMSE as tiebreaker within 0.003 R2",
    "",
    f"Final OOF metrics  : R2={bm['r2']:.4f}  RMSE={bm['rmse']:.4f}  MAE={bm['mae']:.4f}  "
        f"QWK={bm['qwk']:.4f}  Acc@0={bm['acc0']*100:.1f}%  Acc@1={bm['acc1']*100:.1f}%",
    "",
    "Deliverables produced above:",
    "  Sec26a  - full reg/ord/blend metrics table (OOF + 5-fold CV mean+-std)",
    "  Sec27   - confusion matrix (row-normalized, with raw counts annotated)",
    "  Sec28   - frieze visualization(s), sorted by prediction, 8-class-aware",
    "  Sec29   - this per-class statistics table",
    "="*80,
]
print("\n".join(summary_lines))

## 30 · Deployment (updated for the true final config)

§18's `predict_quality()` assumed a dual-head model, since at that point in
the notebook the dual-head + composite-loss recipe was the only thing
tested. §25's ablation could in principle have picked `single_mse` (no dual
head at all) as the winner -- this version checks `FINAL_HEAD_TYPE` and
works either way. **This definition replaces §18's** -- call this one, not
that one, since it also picks up the true encoding/color/filtering settings
from §23-26 (§18's was frozen at the earlier §17 settings).

In [ ]:
def predict_quality(rgb_uint8_image):
    """rgb_uint8_image: HxWx3 uint8 RGB image (already cropped to the produce,
    any size -- resized to IMG_SIZE internally). Returns reg/ord/blend scores
    for a dual-head final model, or a single 'score' for a single-head one --
    averaged across the 5 fold checkpoints from Sec26's true final run."""
    img = cv2.resize(rgb_uint8_image, (IMG_SIZE[1], IMG_SIZE[0]), interpolation=cv2.INTER_AREA)
    enc_img = convert_encoding(img, WINNING_ENCODING)
    feat    = extract_colorspace_features(enc_img) if FINAL_USE_COLOR else np.zeros(FEAT_DIM, dtype=np.float32)

    normed = normalize_channels_01(enc_img, WINNING_ENCODING)
    img_t  = torch.from_numpy(normed.astype(np.float32)).permute(2, 0, 1).unsqueeze(0).to(device)
    feat_t = torch.from_numpy(feat.astype(np.float32)).unsqueeze(0).to(device)

    model = FINAL_LOSS_CONFIG['model_cls'](use_color_feat=FINAL_USE_COLOR).to(device)
    model.eval()

    if FINAL_HEAD_TYPE == 'dual':
        reg_preds, ord_preds, blend_preds = [], [], []
        with torch.no_grad():
            for fold_idx in range(K_FOLDS):
                ckpt = f'training/model/best_FINAL_fold{fold_idx+1}.pt'
                model.load_state_dict(torch.load(ckpt, map_location=device))
                ord_logits, reg_out = model(img_t, feat_t if FINAL_USE_COLOR else None)
                reg_score, ord_score, blend = predict_reg_ord_blend(ord_logits, reg_out, FINAL_ORDINAL_MODE)
                reg_preds.append(reg_score.item()); ord_preds.append(ord_score.item()); blend_preds.append(blend.item())
        return dict(
            reg_score=float(np.mean(reg_preds)),  reg_std=float(np.std(reg_preds)),
            ord_score=float(np.mean(ord_preds)),  ord_std=float(np.std(ord_preds)),
            blend_score=float(np.mean(blend_preds)), blend_std=float(np.std(blend_preds)),
            n_folds_ensembled=K_FOLDS,
        )
    else:
        preds = []
        with torch.no_grad():
            for fold_idx in range(K_FOLDS):
                ckpt = f'training/model/best_FINAL_fold{fold_idx+1}.pt'
                model.load_state_dict(torch.load(ckpt, map_location=device))
                out = model(img_t, feat_t if FINAL_USE_COLOR else None)
                preds.append(out.clamp(1.0, float(N_CLASSES_PRESENT)).item())
        return dict(score=float(np.mean(preds)), score_std=float(np.std(preds)), n_folds_ensembled=K_FOLDS)

print("predict_quality() (final, head-type-aware) ready ✔")
print("Example: predict_quality(X_rgb[0])")